# NadiKampus — Preprocessing Teks untuk NLP & Topic Detection

Notebook ini menyiapkan korpus teks dari dua sumber:

| Sumber | Isi yang diambil |
|---|---|
| **Telegram** (3 file CSV, menfess PENS) | post utama, komentar, dan reply |
| **Google Form** (survei kesejahteraan mahasiswa) | **hanya 5 kolom komentar/pertanyaan terbuka** (skala Likert tidak dibawa) |

Alur: `load → buang duplikat/kosong → samakan skema → cleaning bertingkat → flag kualitas → simpan Parquet`.

Hasil akhirnya satu korpus dengan beberapa versi teks, supaya bisa dipakai model yang berbeda:

| Kolom | Cocok untuk |
|---|---|
| `text_clean` | embedding / IndoBERT / BERTopic / sentimen berbasis transformer (casing & emoji dipertahankan) |
| `text_norm` | lexicon sentimen, TF-IDF (lowercase, tanpa tanda baca, slang dinormalisasi) |
| `text_tokens` | LDA / NMF / c-TF-IDF (stopword dibuang, opsional stemming) |

## 1. Setup & konfigurasi

In [1]:
# Parquet butuh pyarrow. Kalau belum ada, jalankan sekali: 
!pip install pyarrow
# Stemming (opsional) butuh Sastrawi:                      
!pip install Sastrawi


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached Sastrawi-1.0.1-py2.py3-none-any.whl.metadata (909 bytes)
Using cached Sastrawi-1.0.1-py2.py3-none-any.whl (209 kB)



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re, hashlib, unicodedata, urllib.request
from collections import Counter
from functools import lru_cache
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# Parquet butuh pyarrow. Kalau belum ada, jalankan sekali:  %pip install pyarrow
# Stemming (opsional) butuh Sastrawi:                       %pip install Sastrawi

In [ ]:
# ====================== KONFIGURASI ======================
DATA_DIR = Path(".")                    # folder tempat file CSV berada
OUT_DIR  = Path("data")               # folder hasil (parquet)

TELEGRAM_GLOB = "telegram_menfess_*.csv"
GFORM_GLOB    = "SURVEI*KESMA*.csv"      # nama file survei memakai spasi dan tanda kurung

# Minimal jumlah kata (setelah normalisasi) agar teks dianggap layak dipakai.
# Komentar Telegram sering sangat pendek ("nitip", "hmu") -> noise untuk topic detection.
# Jawaban survei yang pendek ("orang tua", "pelatihan") justru informatif -> batas 1.
MIN_WORDS = {"telegram": 3, "gform": 1}

USE_STEMMING            = False   # True = stemming Sastrawi (lambat, tidak perlu untuk BERTopic)
USE_EXTERNAL_SLANG      = True    # unduh kamus slang colloquial-indonesian-lexicon (fallback ke kamus bawaan bila gagal)
KEEP_GFORM_DEMOGRAPHICS = False   # True = bawa gender/angkatan/departemen/prodi/semester (berguna utk filter dashboard)
HASH_SALT               = "nadikampus"   # untuk menganonimkan Sender_ID Telegram

# 5 pertanyaan terbuka GForm -> dimensi (dicocokkan lewat awalan teks kolom)
GFORM_COMMENT_COLS = {
    "Hal apa yang paling membuat Anda tetap bertahan":  "wellbeing",
    "Apa kendala akademik":                              "academic",
    "Apa yang membuat Anda merasa lebih diterima":       "social",
    "Dukungan apa yang paling Anda butuhkan":            "career",
    "Ceritakan secara singkat kesulitan ekonomi":        "economic",
}
GFORM_DEMO_COLS = {
    "Jenis Kelamin": "gender", "Semester saat ini": "semester", "Angkatan": "angkatan",
    "Departemen": "departemen", "Program Studi": "prodi",
}

## 2. Data Telegram

Empat file punya rentang tanggal yang saling beririsan (komentar yang masuk belakangan ikut tercatat di file berikutnya), jadi **duplikat `Message_ID` harus dibuang** setelah digabung.

In [9]:
tg_files = sorted(DATA_DIR.glob(TELEGRAM_GLOB))
assert tg_files, f"Tidak ada file Telegram di {DATA_DIR.resolve()}"

tg_raw = pd.concat(
    [pd.read_csv(f).assign(source_file=f.name) for f in tg_files],
    ignore_index=True,
)
print(f"{len(tg_files)} file -> {len(tg_raw):,} baris mentah")
tg_raw.head(3)

3 file -> 4,807 baris mentah


,Message_ID,Type,Sender_ID,Username,Date_WIB,Text,source_file
0,9530,Main_Post,-1001647181983,pensfess,2026-08-31 23:37:04,"Info kost kost an cowo under 750 yg jarak nya dibawah 1,5 km dari pens, listrik n air include bebas pake, pokoknya y...",telegram_menfess_agustus.csv
1,9529,Main_Post,-1001647181983,pensfess,2026-08-31 22:57:19,Sender punya bisnis yang Alhamdulillah lolos pendanaan PMW nihh 🥹✨ Lagi cari 1 anggota angkatan 2025 buat posisi CCO...,telegram_menfess_agustus.csv
2,9528,Main_Post,-1001647181983,pensfess,2026-08-31 22:54:49,"Ara CE maba, sumpah arek e lucu maneh, gak cocok masuk teknik sampean mbak terlalu pink kwkwk",telegram_menfess_agustus.csv


In [10]:
# --- 2a. buang duplikat Message_ID
n0 = len(tg_raw)
n_conflict = int((tg_raw.groupby("Message_ID")["Text"].nunique(dropna=False) > 1).sum())
tg = tg_raw.drop_duplicates("Message_ID", keep="first").copy()
print(f"Duplikat Message_ID dibuang : {n0 - len(tg):,}  (ID dengan teks berbeda: {n_conflict})")

# --- 2b. tipe & waktu
TYPE_MAP = {"Main_Post": "post", "Comment": "comment", "Reply_To_Thread": "reply"}
tg["doc_type"]  = tg["Type"].map(TYPE_MAP)
tg["timestamp"] = pd.to_datetime(tg["Date_WIB"])          # WIB, tanpa timezone
print(tg["doc_type"].value_counts().to_dict())

Duplikat Message_ID dibuang : 0  (ID dengan teks berbeda: 0)
{'post': 4807}


In [17]:
# --- 2c. thread_id = ID post utama dari setiap komentar/reply (menelusuri rantai reply)
#     Dibuat SEBELUM baris kosong dibuang, supaya rantai reply tidak putus.
#     Untuk post utama, thread_id = Message_ID-nya sendiri (juga bila kolom Reply_To_Msg_ID kosong semua).
parent   = {int(m): (None if pd.isna(p) else int(p)) for m, p in zip(tg["Message_ID"], tg["Reply_To_Msg_ID"])}
post_ids = set(tg.loc[tg["doc_type"] == "post", "Message_ID"].astype(int))

def find_root(mid, max_depth=25):
    cur = int(mid)
    for _ in range(max_depth):
        if cur in post_ids:
            return cur
        nxt = parent.get(cur)
        if nxt is None:
            return None
        cur = nxt
    return None

tg["thread_id"] = pd.array([find_root(m) for m in tg["Message_ID"]], dtype="Int64")
print(f"Thread berhasil dilacak ke post utama: {tg['thread_id'].notna().mean():.1%}")

# --- 2d. anonimkan pengirim (username dibuang, Sender_ID di-hash)
def hash_id(x):
    return hashlib.sha256(f"{HASH_SALT}:{x}".encode()).hexdigest()[:12]

tg["author_id"] = tg["Sender_ID"].map(hash_id)
if tg["author_id"].nunique() <= 1:
    print("[i] Semua pesan dikirim akun channel yang sama -> author_id tidak informatif untuk Telegram.")

# --- 2e. buang teks kosong
tg["text_raw"] = tg["Text"].fillna("").astype(str)
empty = tg["text_raw"].str.strip().eq("")
print(f"Teks kosong dibuang        : {int(empty.sum()):,}")
tg = tg[~empty]

tg_u = pd.DataFrame({
    "doc_id":    "tg_" + tg["Message_ID"].astype(str),
    "source":    "telegram",
    "doc_type":  tg["doc_type"],
    "dimension": None,
    "thread_id": tg["thread_id"],
    "author_id": tg["author_id"],
    "timestamp": tg["timestamp"],
    "text_raw":  tg["text_raw"],
})
print(f"Telegram siap: {len(tg_u):,} dokumen")

Thread berhasil dilacak ke post utama: 100.0%
[i] Semua pesan dikirim akun channel yang sama -> author_id tidak informatif untuk Telegram.
Teks kosong dibuang        : 0
Telegram siap: 4,807 dokumen


## 3. Data Google Form — hanya kolom komentar

Lima pertanyaan terbuka (satu per dimensi) di-*melt* ke format panjang: satu baris = satu jawaban.

In [18]:
gf_path = sorted(DATA_DIR.glob(GFORM_GLOB))
assert gf_path, f"File GForm tidak ditemukan di {DATA_DIR.resolve()}"
gf_raw = pd.read_csv(gf_path[0])
gf_raw.columns = [c.strip() for c in gf_raw.columns]
print(f"{gf_path[0].name}: {gf_raw.shape[0]} responden, {gf_raw.shape[1]} kolom")

gf_raw["timestamp"] = pd.to_datetime(gf_raw["Timestamp"], dayfirst=True)   # format dd/mm/yyyy
gf_raw = gf_raw.sort_values("timestamp").reset_index(drop=True)
gf_raw["respondent_id"] = gf_raw.index + 1

# cocokkan kolom komentar lewat awalan teks
comment_cols = {}
for prefix, dim in GFORM_COMMENT_COLS.items():
    hit = [c for c in gf_raw.columns if c.startswith(prefix)]
    assert len(hit) == 1, f"Kolom untuk '{prefix}' ditemukan {len(hit)}x -> cek nama kolom"
    comment_cols[dim] = hit[0]
pd.Series(comment_cols, name="kolom komentar").to_frame()

SURVEI KESMA (Responses) - Form responses 1.csv: 149 responden, 52 kolom


,kolom komentar
wellbeing,Hal apa yang paling membuat Anda tetap bertahan dan bersemangat menjalani perkuliahan sehari-hari?
academic,Apa kendala akademik yang paling sering Anda alami dan menurut Anda perlu mendapat perhatian kampus?
social,Apa yang membuat Anda merasa lebih diterima dan nyaman berinteraksi di lingkungan kampus?
career,Dukungan apa yang paling Anda butuhkan dari kampus untuk meningkatkan kesiapan menghadapi dunia kerja?
economic,"Ceritakan secara singkat kesulitan ekonomi terbesar yang paling memengaruhi studimu saat ini, dan bantuan apa yang p..."


In [19]:
parts = []
for dim, col in comment_cols.items():
    p = pd.DataFrame({
        "respondent_id": gf_raw["respondent_id"],
        "dimension":     dim,
        "timestamp":     gf_raw["timestamp"],
        "text_raw":      gf_raw[col].fillna("").astype(str),
    })
    if KEEP_GFORM_DEMOGRAPHICS:
        for src, dst in GFORM_DEMO_COLS.items():
            p[dst] = gf_raw[src]
    parts.append(p)

gf_long = pd.concat(parts, ignore_index=True)
empty = gf_long["text_raw"].str.strip().eq("")
print("Jawaban terisi per dimensi:")
print((~empty).groupby(gf_long["dimension"]).sum().to_string())
gf_long = gf_long[~empty]

gf_u = pd.DataFrame({
    "doc_id":    "gf_" + gf_long["respondent_id"].astype(str) + "_" + gf_long["dimension"],
    "source":    "gform",
    "doc_type":  "survey_answer",
    "dimension": gf_long["dimension"],
    "thread_id": pd.array([None] * len(gf_long), dtype="Int64"),
    "parent_id": pd.array([None] * len(gf_long), dtype="Int64"),
    "author_id": "gf_" + gf_long["respondent_id"].astype(str).str.zfill(3),
    "timestamp": gf_long["timestamp"],
    "text_raw":  gf_long["text_raw"],
})
if KEEP_GFORM_DEMOGRAPHICS:
    for dst in GFORM_DEMO_COLS.values():
        gf_u[dst] = gf_long[dst]
print(f"\nGForm siap: {len(gf_u):,} dokumen")

Jawaban terisi per dimensi:
dimension
academic     148
career       147
economic      35
social       148
wellbeing    148

GForm siap: 626 dokumen


In [20]:
corpus = pd.concat([tg_u, gf_u], ignore_index=True)
print(corpus.groupby(["source", "doc_type"], dropna=False).size().to_string())

source    doc_type     
gform     survey_answer     626
telegram  post             4807


## 4. Kamus: slang, stopword, istilah domain

* **Kamus bawaan** (negasi, singkatan umum, istilah kampus) selalu dipakai dan menang atas kamus eksternal.
* **Kamus eksternal** (colloquial-indonesian-lexicon) diunduh sekali ke folder `output/` lalu di-cache.
* `PROTECT` = istilah domain PENS yang tidak boleh diubah oleh kamus mana pun.
* Stopword: daftar bawaan + (jika terpasang) Sastrawi + kata sapaan khas menfess (`nder`, `sender`, `kak`, …). Tambahkan kata lain di `DOMAIN_STOPWORDS` setelah melihat hasil EDA.

In [21]:
PROTECT = {
    "pens", "mypens", "ukt", "kipk", "ethol", "maba", "kating", "bem", "hima", "ormawa",
    "pkkmb", "dosen", "praktikum", "magang", "uts", "uas", "ipk", "sks",
    "dtik", "dte", "dtmk", "dtmk", "lkmm", "wifi",
}

SLANG_BUILTIN = {
    # negasi
    "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak", "enggak": "tidak", "engga": "tidak",
    "gk": "tidak", "tdk": "tidak", "ndak": "tidak", "nda": "tidak", "kagak": "tidak",
    "gaada": "tidak ada", "gada": "tidak ada", "gaperlu": "tidak perlu", "gausah": "tidak usah",
    "gabisa": "tidak bisa", "gamau": "tidak mau", "gatau": "tidak tahu", "gtau": "tidak tahu", "gtw": "tidak tahu",
    "gaenak": "tidak enak", "tau": "tahu",
    # kata fungsi / singkatan umum
    "yg": "yang", "dgn": "dengan", "dg": "dengan", "utk": "untuk", "tuk": "untuk", "buat": "untuk",
    "krn": "karena", "karna": "karena", "tp": "tapi", "sm": "sama", "ama": "sama", "dr": "dari", "dlm": "dalam",
    "klo": "kalau", "kalo": "kalau", "kl": "kalau", "klw": "kalau",
    "udh": "sudah", "udah": "sudah", "dah": "sudah", "sdh": "sudah", "sdah": "sudah",
    "blm": "belum", "blom": "belum", "msh": "masih", "masi": "masih", "sdg": "sedang",
    "bgt": "sangat", "banget": "sangat", "bngt": "sangat",
    "aja": "saja", "aj": "saja", "jg": "juga", "lg": "lagi", "trs": "terus", "trus": "terus",
    "emg": "memang", "emang": "memang", "org": "orang", "orng": "orang", "byk": "banyak", "bnyk": "banyak",
    "jd": "jadi", "jdi": "jadi", "skali": "sekali", "skl": "sekali", "lbh": "lebih", "hrs": "harus",
    "bs": "bisa", "bsa": "bisa", "jgn": "jangan", "dlu": "dulu", "dpt": "dapat", "dapet": "dapat", "dpet": "dapat",
    "ttg": "tentang", "tntg": "tentang", "sampe": "sampai", "smpe": "sampai", "lgsg": "langsung",
    "bkn": "bukan", "sbg": "sebagai", "spy": "supaya", "biar": "supaya",
    "bakal": "akan", "bkal": "akan", "pengen": "ingin", "pgn": "ingin", "mw": "mau",
    "bener": "benar", "bnr": "benar", "bner": "benar",
    "sebenernya": "sebenarnya", "sbnrnya": "sebenarnya",
    # pronomina
    "sy": "saya", "gue": "aku", "gw": "aku", "gua": "aku", "aq": "aku", "ak": "aku",
    "lu": "kamu", "lo": "kamu", "elu": "kamu", "kmu": "kamu",
    # tanya & pembanding
    "gmn": "bagaimana", "gmna": "bagaimana", "gimana": "bagaimana", "knp": "mengapa", "kenapa": "mengapa",
    "kyk": "seperti", "kayak": "seperti", "kek": "seperti", "gitu": "begitu", "gini": "begini", "dikit": "sedikit",
    # waktu
    "skrg": "sekarang", "skrng": "sekarang", "bsk": "besok", "kmrn": "kemarin", "kemaren": "kemarin", "hr": "hari",
    "wkt": "waktu",
    # ucapan
    "mksh": "terimakasih", "makasih": "terimakasih", "makasi": "terimakasih", "thx": "terimakasih",
    "thanks": "terimakasih", "thank": "terimakasih", "tks": "terimakasih", "trims": "terimakasih",
    "tengs": "terimakasih", "terimakasih": "terimakasih",
    "tlg": "tolong", "plis": "tolong", "pls": "tolong", "please": "tolong",
    "yaudah": "ya sudah", "yauda": "ya sudah",
    # variasi umum
    "pake": "pakai", "mls": "malas", "males": "malas", "capek": "lelah", "cape": "lelah",
    "tmn": "teman", "temen": "teman", "tmen": "teman",
    "info": "info", "infokan": "info", "infoin": "info", "infonya": "info",
    # istilah kampus
    "matkul": "mata kuliah", "dsn": "dosen", "mhs": "mahasiswa", "prak": "praktikum", "tgs": "tugas",
    "ae": "saja", "ngapain": "mengapa", "ortu": "orangtua",
    "kost": "kos", "kosan": "kos", "kostan": "kos", "kmps": "kampus", "kmpus": "kampus", "krj": "kerja",
}

# kata Inggris yang di kamus eksternal salah dipetakan ke kata Indonesia
ENGLISH_BLOCK = {"than", "make", "may", "see", "say", "can", "got", "let", "get", "was", "has"}

def load_external_slang():
    url = ("https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/"
           "colloquial-indonesian-lexicon.csv")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    cache = OUT_DIR / "colloquial-indonesian-lexicon.csv"
    try:
        if not cache.exists():
            urllib.request.urlretrieve(url, cache)
        lex = pd.read_csv(cache).dropna(subset=["slang", "formal"])
        lex = lex[lex["slang"].str.fullmatch(r"[a-z]{3,}") & (lex["slang"] != lex["formal"])]
        lex = lex.drop_duplicates("slang")
        lex = lex[~lex["slang"].isin(PROTECT | ENGLISH_BLOCK)]
        return dict(zip(lex["slang"], lex["formal"].str.lower()))
    except Exception as e:
        print(f"[!] Kamus eksternal gagal dimuat ({type(e).__name__}); pakai kamus bawaan saja.")
        return {}

SLANG_EXT = load_external_slang() if USE_EXTERNAL_SLANG else {}
print(f"Kamus bawaan: {len(SLANG_BUILTIN)} entri | kamus eksternal: {len(SLANG_EXT):,} entri")

Kamus bawaan: 166 entri | kamus eksternal: 3,760 entri


In [22]:
BASE_STOPWORDS = set("""
yang di ke dari dan atau ini itu ada adalah untuk dengan pada dalam saya aku kamu dia kita kami kalian mereka anda
tidak bukan belum sudah masih sedang akan bisa dapat mau ingin juga saja lagi lebih sangat paling sekali
dong sih kok deh lah kan nih tuh ya yah kah pun nya jadi karena tapi tetapi namun kalau jika maka agar supaya
oleh sebagai seperti begitu begini sama semua setiap para suatu sebuah seorang saat ketika sambil sampai hingga
dulu tadi nanti mungkin cuma hanya lalu terus biar apa siapa mana mengapa bagaimana berapa kapan dimana
si se ke terimakasih
""".split())

# stopword Inggris minimal (komentar campur Indonesia-Inggris cukup sering muncul)
ENGLISH_STOPWORDS = set("""
the an and or but if of to in on at for with from by as is are was were be been am do does did have has had
this that these those it its you he she we they me my your our their not so just can will would should could
about what when where how why who all any some than then there here also more very too
""".split())

# sapaan & basa-basi khas menfess (sesuaikan setelah melihat hasil EDA)
DOMAIN_STOPWORDS = {
    "nder", "ndr", "sender", "senders", "kak", "kakak", "kk", "mas", "mbak", "mba", "dek", "adek", "rek",
    "bang", "bro", "sis", "guys", "gaes", "gais", "min", "admin", "nitip", "hmu", "spill", "dm",
    "pens", "pensfess",
}

STOPWORDS = set(BASE_STOPWORDS) | ENGLISH_STOPWORDS
try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    STOPWORDS |= set(StopWordRemoverFactory().get_stop_words())
    print("Stopword Sastrawi ikut dipakai.")
except ImportError:
    print("Sastrawi tidak terpasang -> pakai stopword bawaan saja.")
STOPWORDS -= PROTECT
STOPWORDS |= DOMAIN_STOPWORDS      # domain stopword menang atas PROTECT (mis. "pens" sengaja dibuang dari token)
print(f"Total stopword: {len(STOPWORDS)}")

if USE_STEMMING:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    _stemmer = StemmerFactory().create_stemmer()
    @lru_cache(maxsize=None)
    def stem(w):
        return w if w in PROTECT else _stemmer.stem(w)
else:
    def stem(w):
        return w

Stopword Sastrawi ikut dipakai.
Total stopword: 266


## 5. Fungsi cleaning bertingkat

1. **`light_clean`** → `text_clean`: Unicode NFKC, hapus URL & @mention, `#tag` → `tag`, pemendekan huruf berulang (`nderrrr` → `nder`), rapikan spasi. Casing & emoji dipertahankan.
2. **`normalize`** → `text_norm`: lowercase, buang emoji/tanda baca/angka murni, reduplikasi (`teman-teman`, `bner2` → satu kata), tawa (`wkwk`, `hahaha`) dibuang, slang → baku.
3. **`to_tokens`** → `text_tokens`: buang stopword, potong klitik `-nya` pada kata ≥ 6 huruf, stemming opsional.

> Stopword Sastrawi ikut membuang kata negasi (`tidak`, `belum`). Itu wajar untuk topic detection, tapi **untuk analisis sentimen pakai `text_clean`/`text_norm`**, bukan `text_tokens`.

In [23]:
URL_RE          = re.compile(r"(?:https?://|www\.)\S+", re.I)
MENTION_RE      = re.compile(r"@\w+")
HASHTAG_RE      = re.compile(r"#(\w+)")
ZERO_WIDTH_RE   = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
LETTER_REPEAT   = re.compile(r"([^\W\d_])\1{2,}")      # aaa -> a   (angka tidak disentuh)
SYMBOL_REPEAT   = re.compile(r"([^\w\s])\1{2,}")       # !!! / ...... / emoji berulang -> satu
EMOJI_CHAR_RE   = re.compile("[\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF\u2600-\u27BF\u2B50\u2B55]")
EMOJI_ANY_RE    = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF\uFE0F\u200D]+")
LAUGH_RE        = re.compile(r"^(?:(?:wk|kw)+[wk]*|(?:ha|hi|he|hu|ho){2,}h?|xi(?:xi)+x?|awok\w*)$")

def light_clean(text: str) -> str:
    t = unicodedata.normalize("NFKC", text)
    t = ZERO_WIDTH_RE.sub("", t)
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = HASHTAG_RE.sub(r"\1", t)
    t = LETTER_REPEAT.sub(r"\1", t)
    t = SYMBOL_REPEAT.sub(r"\1", t)
    return re.sub(r"\s+", " ", t).strip()

def map_token(tok: str) -> str:
    if tok in PROTECT:
        return tok
    mapped = SLANG_BUILTIN.get(tok) or SLANG_EXT.get(tok) or tok
    # pass kedua dengan kamus bawaan (mis. hasil kamus eksternal "enggak" -> "tidak")
    return " ".join(SLANG_BUILTIN.get(w, w) for w in mapped.split())

def normalize(text: str) -> str:
    t = EMOJI_ANY_RE.sub(" ", text.lower())
    t = re.sub(r"\b([a-z]{3,})-\1\b", r"\1", t)     # teman-teman -> teman
    t = re.sub(r"\b([a-z]{3,})2\b", r"\1", t)       # bner2 -> bner
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    out = []
    for tok in t.split():
        if tok.isdigit() or LAUGH_RE.match(tok):
            continue
        out.extend(w for w in map_token(tok).split() if len(w) > 1)
    return re.sub(r"\borang tua\b", "orangtua", " ".join(out))

def to_tokens(norm: str) -> str:
    toks = []
    for w in norm.split():
        if w in STOPWORDS:
            continue
        if len(w) >= 6 and w.endswith("nya") and w not in PROTECT:
            w = w[:-3]
            if w in STOPWORDS:
                continue
        toks.append(stem(w))
    return " ".join(toks)

# uji cepat
for s in ["Nderrrr, kalo ada info kosan murah dkt PENS dong 😭😭😭 https://t.co/xyz @budi #ukt",
          "wkwkwk gak ada yg mau bantuin tugas2 temen-temen, capek bgt"]:
    c = light_clean(s); n = normalize(c)
    print(f"RAW   : {s}\nCLEAN : {c}\nNORM  : {n}\nTOKENS: {to_tokens(n)}\n")

RAW   : Nderrrr, kalo ada info kosan murah dkt PENS dong 😭😭😭 https://t.co/xyz @budi #ukt
CLEAN : Nder, kalo ada info kosan murah dkt PENS dong 😭 ukt
NORM  : nder kalau ada info kos murah dkt pens dong ukt
TOKENS: info kos murah dkt ukt

RAW   : wkwkwk gak ada yg mau bantuin tugas2 temen-temen, capek bgt
CLEAN : wkwkwk gak ada yg mau bantuin tugas2 temen-temen, capek bgt
NORM  : tidak ada yang mau bantuin tugas teman lelah sangat
TOKENS: bantuin tugas teman lelah



In [24]:
corpus["text_clean"]  = corpus["text_raw"].map(light_clean)
corpus["text_norm"]   = corpus["text_clean"].map(normalize)
corpus["text_tokens"] = corpus["text_norm"].map(to_tokens)

corpus["n_words"]  = corpus["text_norm"].str.split().str.len()
corpus["n_tokens"] = corpus["text_tokens"].str.split().str.len()
emojis = corpus["text_raw"].map(EMOJI_CHAR_RE.findall)
corpus["n_emoji"] = emojis.str.len()
corpus["emojis"]  = emojis.str.join("")
print("Cleaning selesai.")

Cleaning selesai.


## 6. Flag kualitas

Tidak ada baris yang dihapus diam-diam — semuanya diberi flag, lalu `is_valid` dipakai untuk membuat korpus siap-model.

* `is_no_answer` — jawaban kosong makna: `-`, `tidak ada`, `belum ada`, `aman`, dst. (dicocokkan penuh, jadi keluhan seperti *"tidak ada dosen yang peduli"* **tidak** terbuang).
* `is_dup_text` — teks identik (setelah normalisasi) pada **Telegram** (spam / copy-paste). Di GForm duplikat dibiarkan karena jawaban yang sama dari responden berbeda adalah sinyal frekuensi (mis. "orang tua").
* `is_valid` — `n_words` ≥ `MIN_WORDS[source]`, bukan no-answer, bukan duplikat, dan punya minimal 1 token.

In [25]:
NO_ANSWER_EXACT = {
    "", "tidak ada", "belum ada", "tidak tahu", "tidak", "aman", "cukup", "nihil", "biasa saja",
    "sudah cukup", "sudah baik", "baik", "baik baik saja", "lancar", "semua aman",
    "tidak ada kendala", "tidak ada masalah", "tidak ada saran", "belum ada saran",
}
NEG_START = re.compile(r"^(?:tidak|belum) (?:ada|punya|tahu|pernah|merasa|mengalami)\b(.*)$")
NO_ANSWER_FILLER = {
    "ada", "kendala", "masalah", "kesulitan", "saran", "apa", "apapun", "hal", "sama", "sekali", "saat", "ini",
    "sejauh", "yang", "berarti", "paling", "sering", "saya", "alami", "mengalami", "akademik", "untuk",
    "kampus", "ditemui", "dirasakan", "ide", "kepikiran", "merasa", "tahu", "punya", "pernah",
}

def is_no_answer(norm: str) -> bool:
    if norm in NO_ANSWER_EXACT:
        return True
    m = NEG_START.match(norm)
    return bool(m) and all(w in NO_ANSWER_FILLER for w in m.group(1).split())

corpus["is_no_answer"] = corpus["text_norm"].map(is_no_answer)
corpus["is_dup_text"]  = (
    corpus["source"].eq("telegram")
    & corpus["text_norm"].ne("")
    & corpus.duplicated(["source", "text_norm"], keep="first")
)
corpus["is_valid"] = (
    (corpus["n_words"] >= corpus["source"].map(MIN_WORDS))
    & (corpus["n_tokens"] > 0)
    & ~corpus["is_no_answer"]
    & ~corpus["is_dup_text"]
)

summary = (corpus.groupby(["source", "doc_type"], dropna=False)
                 .agg(total=("doc_id", "size"), valid=("is_valid", "sum"),
                      no_answer=("is_no_answer", "sum"), dup=("is_dup_text", "sum"))
                 .assign(valid_pct=lambda d: (d.valid / d.total * 100).round(1)))
summary

,,total,valid,no_answer,dup,valid_pct
source,doc_type,,,,,
gform,survey_answer,626,553,70,0,88.3
telegram,post,4807,4789,3,8,99.6


## 7. Sanity check hasil cleaning

In [26]:
valid = corpus[corpus["is_valid"]]

print("Contoh sebelum -> sesudah (acak):")
for _, r in valid.sample(8, random_state=7).iterrows():
    print(f"[{r.source}/{r.doc_type}]")
    print("  raw   :", r.text_raw[:150].replace("\n", " "))
    print("  norm  :", r.text_norm[:150])
    print("  tokens:", r.text_tokens[:150], "\n")

Contoh sebelum -> sesudah (acak):
[telegram/post]
  raw   : Info manga/vn yang menurut mu seruh buat di baca, sender lagi kehabisan manga buat dibaca
  norm  : info manga vn yang menurut mu seruh untuk di baca sender lagi kehabisan manga untuk dibaca
  tokens: info manga vn mu seruh baca kehabisan manga dibaca 

[gform/survey_answer]
  raw   : Ethol
  norm  : ethol
  tokens: ethol 

[telegram/post]
  raw   : Saran kegiatan setelah SKEM kelar dong 😔 yang bisa nambah uang kalau bisa biar ada tambahan buat pergi menghilang dari surabaya lagi
  norm  : saran kegiatan setelah skem kelar dong yang bisa bertambah uang kalau bisa supaya ada tambahan untuk pergi menghilang dari surabaya lagi
  tokens: saran kegiatan skem kelar bertambah uang tambahan pergi menghilang surabaya 

[telegram/post]
  raw   : Rek info wisuda tahun ini fixnya ada yang tau gak di tanggal berapa?
  norm  : rek info wisuda tahun ini fixnya ada yang tahu tidak di tanggal berapa
  tokens: info wisuda tahun fix tahu tanggal

In [27]:
# token terbanyak per sumber -> dasar untuk menambah DOMAIN_STOPWORDS
for src, g in valid.groupby("source"):
    cnt = Counter(w for s in g["text_tokens"] for w in s.split())
    print(f"=== {src} ({len(g):,} dokumen) ===")
    print(", ".join(f"{w}({c})" for w, c in cnt.most_common(40)), "\n")

=== gform (553 dokumen) ===
teman(110), kampus(56), dosen(56), banyak(48), orangtua(47), mahasiswa(39), kuliah(36), tugas(36), kerja(32), baik(30), kurang(28), informasi(28), pelatihan(28), merasa(27), belajar(26), praktikum(25), lingkungan(24), materi(23), orang(22), waktu(22), beasiswa(21), membuat(20), dukungan(20), kelas(20), masa(19), nyaman(19), magang(18), terlalu(17), membantu(17), fasilitas(17), sering(17), depan(16), dunia(16), cukup(16), uang(15), tetap(15), hari(15), keluarga(15), ilmu(15), selalu(15) 

=== telegram (4,789 dokumen) ===
info(719), teman(309), kampus(303), tahu(283), tanya(262), punya(246), tempat(231), orang(229), maba(219), pakai(208), hari(198), kos(190), jam(177), kira(172), laptop(165), beli(163), jangan(159), kuliah(155), cara(145), buka(140), butuh(139), parkiran(137), adakah(134), sekarang(130), mahasiswa(128), pernah(128), baru(128), memang(126), pas(124), kemarin(123), banyak(123), malah(121), anak(119), jual(118), soal(116), lihat(116), semester(11

In [28]:
# kata yang paling sering diubah oleh normalisasi slang -> cek apakah ada pemetaan yang aneh
sample = corpus["text_clean"].sample(min(4000, len(corpus)), random_state=1)
changed = Counter()
for s in sample:
    for tok in set(re.findall(r"[a-z]+", s.lower())):
        new = map_token(tok)
        if new != tok:
            changed[(tok, new)] += 1
pd.DataFrame([(a, b, c) for (a, b), c in changed.most_common(40)], columns=["asli", "menjadi", "frekuensi"])

,asli,menjadi,frekuensi
0,ga,tidak,490
1,yg,yang,476
2,buat,untuk,407
3,udah,sudah,255
4,kalo,kalau,236
5,aja,saja,218
6,banget,sangat,183
7,gak,tidak,173
8,tau,tahu,171
9,gimana,bagaimana,142


In [29]:
valid.groupby("source")["n_words"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
source,,,,,,,,
gform,553.0,11.3,16.4,1.0,3.0,6.0,14.0,218.0
telegram,4789.0,16.5,8.7,3.0,10.0,14.0,22.0,49.0


## 8. Simpan ke Parquet

Dua file:

* `nadikampus_corpus_all.parquet` — semua dokumen + flag (untuk audit / EDA).
* `nadikampus_corpus_model.parquet` — hanya `is_valid == True`, **siap masuk model**.

In [30]:
cols = ["doc_id", "source", "doc_type", "dimension", "thread_id", "parent_id", "author_id", "timestamp"]
if KEEP_GFORM_DEMOGRAPHICS:
    cols += list(GFORM_DEMO_COLS.values())
cols += ["text_raw", "text_clean", "text_norm", "text_tokens",
         "n_words", "n_tokens", "n_emoji", "emojis",
         "is_no_answer", "is_dup_text", "is_valid"]

out = corpus[cols].copy()
for c in ["source", "doc_type", "dimension"]:
    out[c] = out[c].astype("category")

OUT_DIR.mkdir(parents=True, exist_ok=True)
p_all, p_model = OUT_DIR / "nadikampus_corpus_all.parquet", OUT_DIR / "nadikampus_corpus_model.parquet"
out.to_parquet(p_all, index=False)
out[out["is_valid"]].to_parquet(p_model, index=False)

for p in (p_all, p_model):
    df = pd.read_parquet(p)
    print(f"{p.name:36s} {df.shape[0]:>6,} baris x {df.shape[1]} kolom  {p.stat().st_size/1024:,.0f} KB")

nadikampus_corpus_all.parquet         5,433 baris x 19 kolom  1,462 KB
nadikampus_corpus_model.parquet       5,342 baris x 19 kolom  1,458 KB


In [31]:
pd.read_parquet(p_model).head()

,doc_id,source,doc_type,dimension,thread_id,parent_id,author_id,timestamp,text_raw,text_clean,text_norm,text_tokens,n_words,n_tokens,n_emoji,emojis,is_no_answer,is_dup_text,is_valid
0,tg_9530,telegram,post,NaN,9530,<NA>,391e97a9f824,2026-08-31 23:37:04,"Info kost kost an cowo under 750 yg jarak nya dibawah 1,5 km dari pens, listrik n air include bebas pake, pokoknya y...","Info kost kost an cowo under 750 yg jarak nya dibawah 1,5 km dari pens, listrik n air include bebas pake, pokoknya y...",info kos kos an cowok under yang jarak nya dibawah km dari pens listrik air include bebas pakai pokoknya yang worth ...,info kos kos cowok under jarak dibawah km listrik air include bebas pakai pokok worth teman pindahh kos ku sekarang ...,36,23,0,,False,False,True
1,tg_9529,telegram,post,NaN,9529,<NA>,391e97a9f824,2026-08-31 22:57:19,Sender punya bisnis yang Alhamdulillah lolos pendanaan PMW nihh 🥹✨ Lagi cari 1 anggota angkatan 2025 buat posisi CCO...,Sender punya bisnis yang Alhamdulillah lolos pendanaan PMW nihh 🥹✨ Lagi cari 1 anggota angkatan 2025 buat posisi CCO...,sender punya bisnis yang alhamdulillah lolos pendanaan pmw nih lagi cari anggota angkatan untuk posisi cco terutama ...,punya bisnis alhamdulillah lolos pendanaan pmw cari anggota angkatan posisi cco terutama suka desain konten tertarik...,29,18,3,🥹✨🫶,False,False,True
2,tg_9528,telegram,post,NaN,9528,<NA>,391e97a9f824,2026-08-31 22:54:49,"Ara CE maba, sumpah arek e lucu maneh, gak cocok masuk teknik sampean mbak terlalu pink kwkwk","Ara CE maba, sumpah arek e lucu maneh, gak cocok masuk teknik sampean mbak terlalu pink kwkwk",ara ce maba sumpah arek lucu maneh tidak cocok masuk teknik sampean mbak terlalu pink,ara ce maba sumpah arek lucu maneh cocok masuk teknik sampean terlalu pink,15,13,0,,False,False,True
3,tg_9527,telegram,post,NaN,9527,<NA>,391e97a9f824,2026-08-31 22:53:57,Ada yang jual sepatu futsal gak?,Ada yang jual sepatu futsal gak?,ada yang jual sepatu futsal tidak,jual sepatu futsal,6,3,0,,False,False,True
4,tg_9526,telegram,post,NaN,9526,<NA>,391e97a9f824,2026-08-31 22:01:30,"infoin dong oprec DPM kapan ya, sistem pemilihannya gimana","infoin dong oprec DPM kapan ya, sistem pemilihannya gimana",info dong oprec dpm kapan ya sistem pemilihannya bagaimana,info oprec dpm sistem pemilihan,9,5,0,,False,False,True


### Pemakaian di tahap berikutnya
pemodelan
